# Step 6: Persist sightings

![Step 6 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/06-step.png)

## What you're building today

A **database** that remembers every bird the watcher has seen.

Right now we look at a frame, identify the bird, and forget it 5 seconds later when the loop moves on. That's sad — what if a Pileated Woodpecker shows up at 6am and you're not awake yet? You'd never know.

Today we save sightings to a SQLite database: timestamp, species, confidence, snapshot path. Then we can ask the database questions like "what species showed up this week?".

By the end you'll be able to run a SQL query and see your sighting history.

## Step 6.1 — What is SQLite?

SQLite is a database that lives in a single file. There's no server, no login, no setup. It's just a `.db` file you can copy around like any other file.

It comes with Python — no `pip install` needed. Python has a built-in `sqlite3` module.

We use SQL (a query language) to talk to it. SQL looks like English sentences with a fixed structure:

```sql
SELECT species, confidence FROM sightings WHERE timestamp > '2026-07-01';
```

In [ ]:
import sqlite3
from pathlib import Path
from datetime import datetime

DB_PATH = Path("data/birds.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

def get_conn():
    """Open a connection. Each call is a fresh handle."""
    return sqlite3.connect(DB_PATH)

# Create the schema if it doesn't exist
with get_conn() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS sightings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT NOT NULL,
            species TEXT NOT NULL,
            confidence REAL NOT NULL,
            snapshot_path TEXT,
            bbox_x1 INTEGER, bbox_y1 INTEGER,
            bbox_x2 INTEGER, bbox_y2 INTEGER
        )
    """)
    conn.commit()

print(f"Database: {DB_PATH} ({DB_PATH.stat().st_size} bytes)")

## Step 6.2 — Insert a sighting

The `INSERT INTO` statement adds a row. We use parameterized queries (the `?` placeholders) so SQL injection is impossible — even if the species name has a quote in it.

In [ ]:
def record_sighting(
    species: str,
    confidence: float,
    snapshot_path: str = "",
    bbox: tuple[int, int, int, int] = (0, 0, 0, 0),
) -> int:
    """Save one sighting to the database. Returns the new row id."""
    with get_conn() as conn:
        cur = conn.execute(
            """
            INSERT INTO sightings
                (timestamp, species, confidence, snapshot_path,
                 bbox_x1, bbox_y1, bbox_x2, bbox_y2)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (datetime.now().isoformat(timespec="seconds"),
             species, confidence, snapshot_path, *bbox),
        )
        conn.commit()
        return cur.lastrowid

# Add three fake sightings so we have something to query
record_sighting("American Robin", 0.92, "data/snapshots/r1.jpg", (10, 20, 200, 300))
record_sighting("Northern Cardinal", 0.87, "data/snapshots/r2.jpg", (50, 60, 250, 350))
record_sighting("American Robin", 0.79, "data/snapshots/r3.jpg", (30, 40, 220, 320))
print("Inserted 3 test sightings")

## Step 6.3 — Query the database

Now let's ask questions:

In [ ]:
with get_conn() as conn:
    print("=== all sightings ===")
    for row in conn.execute("SELECT * FROM sightings ORDER BY timestamp DESC"):
        print(f"  #{row[0]}  {row[1]}  {row[2]:25s}  ({row[3]:.2f})")

    print("\n=== species counts ===")
    for row in conn.execute(
        "SELECT species, COUNT(*) AS n FROM sightings GROUP BY species ORDER BY n DESC"
    ):
        print(f"  {row[0]:25s}  {row[1]}")

    print("\n=== high confidence only (>0.85) ===")
    for row in conn.execute(
        "SELECT timestamp, species, confidence FROM sightings WHERE confidence > 0.85"
    ):
        print(f"  {row[0]}  {row[1]:25s}  ({row[2]:.2f})")

## Step 6.4 — Wire it into the pipeline

Now: every time the bird detector finds something AND the species classifier names it, save it. Reuse the loop from notebook 03.

In [ ]:
import time, requests, os
from dotenv import load_dotenv
from PIL import Image
from ultralytics import YOLO
from pathlib import Path

load_dotenv()

RUNNING_IN_COLAB = "COLAB_GPU" in os.environ
SOURCE = (
    "https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/data/samples/sample-bird.jpg"
    if RUNNING_IN_COLAB
    else f"http://{os.environ.get('BIRD_WATCHER_PHONE_IP', '192.168.1.42')}:8080/photo.jpg"
)

detector = YOLO("yolov8n.pt")
MAX_FRAMES = 3

for i in range(MAX_FRAMES):
    frame_path = Path(f"data/snapshots/poll-{i:03d}.jpg")
    frame_path.write_bytes(requests.get(SOURCE, timeout=10).content)

    results = detector(frame_path, verbose=False)
    bird_boxes = [b for b in results[0].boxes if int(b.cls[0]) == 14 and float(b.conf[0]) > 0.4]

    if not bird_boxes:
        print(f"[{i+1}/{MAX_FRAMES}] no bird")
    else:
        box = bird_boxes[0]
        bbox = tuple(int(v) for v in box.xyxy[0])
        crop = Image.open(frame_path).crop(bbox)
        species = classifier(crop)[0]
        record_sighting(species["label"], species["score"], str(frame_path), bbox)
        print(f"[{i+1}/{MAX_FRAMES}] saved {species['label']} ({species['score']:.2f})")

    if i < MAX_FRAMES - 1:
        time.sleep(3)

print("\nDone — all sightings persisted to", DB_PATH)

## Done?

If your query in Step 6.3 shows real rows (the 3 you inserted + however many the loop added), the library is wired up. 🎉

Try this query in a new cell:

```python
import sqlite3
with sqlite3.connect(DB_PATH) as conn:
    for row in conn.execute("SELECT * FROM sightings"):
        print(row)
```

## What's next

Issue **#7: Slack notifications**. Saving is great, but you want to know in real time when a Pileated Woodpecker shows up. Next we wire up a Slack ping. 🐦